In [6]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

#Load dataset

df = pd.read_excel(r"C:\Users\gangu\Desktop\Project-demo\customer-segmentation-unsupervised\data\raw\Online Retail.xlsx")

print("Initial shape:", df.shape)
print(df.head())



Initial shape: (541909, 8)
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  


In [10]:

# 2. Basic Cleaning

df = df.dropna(subset=['CustomerID'])

# Convert data types
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Remove negative or zero quantities 
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

print("After cleaning:", df.shape)


# 3. Handle Missing Values


# Description missing values → replace with 'Unknown'
df['Description'] = df['Description'].fillna("Unknown")


# 4. Outlier Detection & Treatment

# Using IQR method for Quantity and UnitPrice

def remove_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    return data[(data[column] >= lower) & (data[column] <= upper)]

df = remove_outliers(df, 'Quantity')
df = remove_outliers(df, 'UnitPrice')

print("After outlier removal:", df.shape)


# 5. Feature Creation (Base)


# Total transaction value
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# 6. RFM Feature Engineering


# Reference date (latest date in dataset)
reference_date = df['InvoiceDate'].max()

# Group by CustomerID
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,   # Recency
    'InvoiceNo': 'count',                                      # Frequency
    'TotalPrice': 'sum',                                       # Monetary
    'Quantity': 'sum'                                          # Total Quantity
})

rfm.columns = ['Recency', 'Frequency', 'Monetary', 'Total_Quantity']
df.isnull().sum()


After cleaning: (338151, 9)
After outlier removal: (322735, 9)


InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
TotalPrice     0
dtype: int64

In [11]:

# 7. Behavioral Features

# Average order value
rfm['Avg_Order_Value'] = rfm['Monetary'] / rfm['Frequency']

# Purchase frequency (time gap between purchases)
first_purchase = df.groupby('CustomerID')['InvoiceDate'].min()
last_purchase = df.groupby('CustomerID')['InvoiceDate'].max()

rfm['Purchase_Frequency'] = (
    (last_purchase - first_purchase).dt.days / rfm['Frequency']
)


# 8. Derived Features (Important)


# Spending rate (money per item)
rfm['Spending_per_Item'] = rfm['Monetary'] / rfm['Total_Quantity']

# Customer value score (simple combined metric)
rfm['Customer_Value'] = (
    rfm['Monetary'] * rfm['Frequency']
)

# 9. Handle Missing (after feature engineering)


rfm = rfm.fillna(0)


In [ ]:


# 10. Feature Scaling


scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm)

rfm_scaled = pd.DataFrame(rfm_scaled, index=rfm.index, columns=rfm.columns)

print("\nFinal Features:")
print(rfm.head())

print("\nScaled Data Sample:")
print(rfm_scaled.head())


# 11. Save Processed Data

rfm.to_csv("processed_customer_data.csv")
rfm_scaled.to_csv("scaled_customer_data.csv")

print("\nData preprocessing completed successfully.")



Final Features:
            Recency  Frequency  Monetary  Total_Quantity  Avg_Order_Value  \
CustomerID                                                                  
12347.0           1        164   3243.33            1881        19.776402   
12348.0         248          6     90.20             140        15.033333   
12349.0          18         53    918.10             510        17.322642   
12350.0         309         16    294.40             196        18.400000   
12352.0          35         59    986.34             476        16.717627   

            Purchase_Frequency  Spending_per_Item  Customer_Value  
CustomerID                                                         
12347.0               2.225610           1.724258       531906.12  
12348.0              18.166667           0.644286          541.20  
12349.0               0.000000           1.800196        48659.30  
12350.0               0.000000           1.502041         4710.40  
12352.0               4.406780     

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'TotalPrice'],
      dtype='object')